In [ ]:
## Exercise: gradient
##
## Continues sessions/softmax_regression.ipynb, and uses W, X_test, Y_test,
## gradient, jax and np from it.
##
## We want the gradient of the negative log-likelihood
##
##     NLL(W) = -(1/n) * sum_i sum_k Y_ik * log(Yhat_ik),    Yhat = softmax(X @ W)
##
## As the tip in the exercise says, this is the logistic regression gradient
## again. Differentiating the softmax and the log separately is unpleasant, but
## the two cancel almost completely, and what survives is
##
##     d NLL / d Z = (Yhat - Y) / n
##
## Then Z = X @ W gives one more chain-rule step:
##
##     d NLL / d W = X.T @ (Yhat - Y) / n
##
## So the gradient is the inputs weighted by how wrong the prediction was. That
## (Yhat - Y) error term is exactly what appeared in logistic regression - the
## softmax model is the same model with more than two classes.

def mygradient(W, X, Y):
    Z = X @ W                      # (n, 784) @ (784, 10) -> (n, 10)
    Yhat = jax.nn.softmax(Z)
    return X.T @ (Yhat - Y) / X.shape[0]   # (784, n) @ (n, 10) -> (784, 10)


In [ ]:
## Check against jax.grad

assert np.allclose(mygradient(W, X_test, Y_test), gradient(W, X_test, Y_test), atol=1e-5)

difference = abs(mygradient(W, X_test, Y_test) - gradient(W, X_test, Y_test)).max()
print('matches jax.grad; largest absolute difference {:.2e}'.format(difference))


### Why bother, when `jax.grad` already does it?

Two things worth taking from this exercise.

The analytic form is cheap to reason about. It says the update direction is a sum of
inputs weighted by prediction error, so a confidently wrong example moves the weights a
long way and an already-correct one barely moves them at all. You cannot read that off
`jax.grad(NLL)`.

It is also a check on the *model*, not just on the code. If the analytic gradient and the
autodiff gradient disagree, one of the two is wrong - and early in a derivation it is
usually the analytic one, which is how you discover that the loss you wrote is not the
loss you meant.
